In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
import pickle
import numpy as np

In [2]:
data_path = '../results/FP_Truthful_Oracle_sigmoids_linucb_gaus_emb_prova/agent_stats_run_0_ctr_0.97_alpha_1.csv'

In [3]:
data = pd.read_csv(data_path)

In [15]:
y_df = data[['publisher', 'clicks', 'impressions', 'Iteration']]

In [5]:
embedding_path = '../src/publisher_embedding/data/embeddings_to_pick/pub_gaus_emb.pkl'

In [6]:
embeddings = pickle.load(open(embedding_path, 'rb'))

In [10]:
def dict_to_dataframe(dict):
    embedding_dim = list(dict.values())[0].shape[0]
    columns_list = ['publisher'] + [f'dim_{i}' for i in range(embedding_dim)]
    df = pd.DataFrame(columns=columns_list)
    for key, value in dict.items():
        df = pd.concat(
            [
                df, 
                pd.DataFrame([
                    [key] + list(value)
                ], columns=columns_list)]
            ,
            ignore_index=True
        )
    return df

In [11]:
embeddings_df = dict_to_dataframe(embeddings)

/var/folders/pt/p9_0myf16lx1rjjrf63vkx5w0000gp/T/ipykernel_4196/3271154136.py:6: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(


In [12]:
training_data = pd.merge(
    y_df, 
    embeddings_df, 
    on='publisher',
    how='left'
)

In [13]:
lin_reg_click = LinearRegression()
lin_reg_imp = LinearRegression()

X = training_data.drop(columns=['publisher', 'clicks', 'impressions'])
y_1 = training_data['clicks']
y_2 = training_data['impressions']

lin_reg_click.fit(X, y_1)
lin_reg_imp.fit(X, y_2)

LinearRegression()

In [14]:
y_pred_click = lin_reg_click.predict(X)
y_pred_imp = lin_reg_imp.predict(X)

In [17]:
data_with_pred = data.copy()
data_with_pred['clicks_pred'] = y_pred_click
data_with_pred['impressions_pred'] = y_pred_imp

In [18]:
data_with_pred = data_with_pred[data_with_pred['Iteration'] > 0]
data_with_pred = data_with_pred[['Iteration', 'publisher', 'clicks', 'est_clicks', 'clicks_pred', 'impressions', 'est_impressions', 'impressions_pred']] 